# Analysis workflow: impute, fit, pool

Multiple imputation accounts for the uncertainty about the missing values in the final inference:

1. impute several completed datasets,
2. fit the analysis model to each,
3. pool the results with Rubin's rules.

This is R's classic `nhanes` example:
`imp <- mice(nhanes, m = 10)`, `fit <- with(imp, lm(chl ~ age + bmi))`, `summary(pool(fit))`.

## 1. Imports and data

In [1]:
import pandas as pd

from imputation import MICE

df = pd.read_csv("data/nhanes.csv")
print(df.isna().sum())

age     0
bmi     9
hyp     8
chl    10
dtype: int64


## 2. Impute

In [2]:
mice = MICE(df)
mice.impute(n_imputations=10, seed=2024)
print(f"{len(mice.imputed_datasets)} completed datasets of shape {mice.imputed_datasets[0].shape}")

10 completed datasets of shape (25, 4)


## 3. Fit the model to each dataset

`fit` takes a formula in statsmodels/patsy syntax and fits ordinary least squares by default
(`model=` accepts other statsmodels formula models, such as `smf.logit` or `smf.glm`).

In [3]:
mice.fit("chl ~ age + bmi")
print(f"Fitted {len(mice.model_results)} models")
print("Estimates in the first dataset:")
print(mice.model_results[0].params)

Fitted 10 models
Estimates in the first dataset:
Intercept   -46.429352
age          33.073178
bmi           6.617432
dtype: float64


## 4. Pool with Rubin's rules

`pool()` returns a dictionary. Its `"table"` holds R's pooled table: the pooled `estimate`, the
within (`ubar`), between (`b`) and total (`t`) variance, the Barnard-Rubin degrees of freedom
`df`, `riv`, `lambda`, the fraction of missing information `fmi`, and the t-based standard error,
p-value and 95% interval.

In [4]:
result = mice.pool()
table = result["table"]
table[["estimate", "std_error", "df", "p_value", "conf_low", "conf_high", "fmi"]].round(3)

,estimate,std_error,df,p_value,conf_low,conf_high,fmi
term,,,,,,,
Intercept,-14.095,74.500,10.901,0.853,-178.249,150.060,0.453
age,35.153,10.542,13.523,0.005,12.468,57.839,0.348
bmi,5.552,2.372,10.918,0.039,0.327,10.777,0.452


The same results as a printable summary:

In [5]:
print(mice.pool(summ=True))

                               Results: MICE
Dependent variable:              chl              Scale              1108.16
Sample size:                     25               M                  10     
----------------------------------------------------------------------------
           Coef.   Std.Err.    t    P>|t|    [0.025   0.975]     df    FMI  
----------------------------------------------------------------------------
Intercept -14.0946  74.4998 -0.1892 0.8534 -178.2490 150.0598 10.9013 0.4526
age        35.1532  10.5419  3.3346 0.0051   12.4679  57.8385 13.5227 0.3476
bmi         5.5520   2.3717  2.3410 0.0393    0.3272  10.7767 10.9181 0.4519



## Summary

1. `impute` created 10 completed datasets.
2. `fit` fitted the regression to each.
3. `pool` combined the fits: the standard errors include the variation between the datasets, and
   `fmi` shows how much of the information about each coefficient is missing.

With 25 rows the degrees of freedom are small and the intervals wide; this is the data, not the
method.

Next: `05_diagnostic_plots.ipynb` to check the imputations visually.